# QnA_using_RNN

# Import Libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Import Dataset

In [4]:
df = pd.read_csv('100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


# Preprocessing Text

In [5]:
def preprocessing(text: str) -> str:
  text = text.lower()
  text = text.replace("?", "")
  text = text.replace("'", "")
  return text

# Tokenize Text

In [6]:
def tokenized_text(text: str) -> list:
  text = preprocessing(text)
  return text.split()

In [8]:
tokenized_text("I' Love You-Jaan?")

['i', 'love', 'you-jaan']

In [9]:
df.columns

Index(['question', 'answer'], dtype='object')

# Building Vocabulary

In [22]:
vocabulary = {
    '<UNK>': 0
}

In [19]:
def build_vocab(row):
  question_tokenized = tokenized_text(row['question'])
  answer_tokenized = tokenized_text(row['answer'])

  merge_text = question_tokenized + answer_tokenized

  for vocab in merge_text:

    if vocab not in vocabulary:
      vocabulary[vocab] = len(vocabulary)

In [25]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [26]:
len(vocabulary)

324

# Convert Text Sentense to Integer

In [28]:
def text_to_indeces(text: str, vocab: dict):

  texted_token = []

  tokenized = tokenized_text(text)
  for token in tokenized:
    if token in vocabulary:
      texted_token.append(vocabulary[token])
    else:
      texted_token.append(vocabulary["<UNK>"])
  return texted_token

In [34]:
text_to_indeces("What is Machine-Learning", vocabulary)

[1, 2, 0]

In [35]:
vocabulary

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

# Dataset Creation

In [43]:
class QnADataset(Dataset):
    def __init__(self, df: pd.DataFrame, vocabulary: dict) -> None:
        super().__init__()
        self.df = df.reset_index(drop=True)  # ensure clean indexing
        self.vocabulary = vocabulary

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        question = self.df.iloc[index]["question"]
        answer = self.df.iloc[index]["answer"]

        # Convert text to index list
        numeric_question = text_to_indeces(question, self.vocabulary)
        numeric_answer = text_to_indeces(answer, self.vocabulary)

        # Convert to tensors (make them LongTensor for embedding layers)
        question_tensor = torch.tensor(numeric_question, dtype=torch.long)
        answer_tensor = torch.tensor(numeric_answer, dtype=torch.long)

        return question_tensor, answer_tensor

In [40]:
index=7
text_to_indeces(df.iloc[index]['question'], vocabulary)

[1, 2, 3, 37, 38, 39, 40]

In [44]:
dataset = QnADataset(df, vocabulary)

# DataLoader Load

In [45]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [47]:
for q, a in dataloader:
  print(q)
  print(a)
  print('-'*45)

tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]])
tensor([[199]])
---------------------------------------------
tensor([[ 42, 137,   2, 138,  39, 175, 269]])
tensor([[99]])
---------------------------------------------
tensor([[ 42, 290, 291, 118, 292, 158, 293, 294]])
tensor([[295]])
---------------------------------------------
tensor([[ 1,  2,  3, 92, 93, 94]])
tensor([[95]])
---------------------------------------------
tensor([[ 78,  79, 150, 151,  14, 152, 153]])
tensor([[154]])
---------------------------------------------
tensor([[  1,   2,   3,  69,   5, 155]])
tensor([[156]])
---------------------------------------------
tensor([[  1,   2,   3,   4,   5, 206]])
tensor([[207]])
---------------------------------------------
tensor([[42, 86, 87, 88, 89, 39, 90]])
tensor([[91]])
---------------------------------------------
tensor([[ 1,  2,  3, 33, 34,  5, 35]])
tensor([[36]])
---------------------------------------------
tensor([[10, 29,  3, 30, 31]])
tensor([[32]])
------

# Simple RNN

In [58]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super(SimpleRNN, self).__init__()

        """
        we cannot use sequential
        sequential expect one input
        but rnn gives two output
        """

        self.embedding = nn.Embedding(vocab_size, 50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.output = nn.Linear(64, vocab_size)

    def forward(self, question):
        # question: (batch, seq_len)
        embedded = self.embedding(question)               # (batch, seq_len, 50)
        out, hidden = self.rnn(embedded)                  # hidden: (1, batch, 64)
        hidden = hidden.squeeze(0)                       # (batch, 64)
        logits = self.output(hidden)                      # (batch, vocab_size)
        return logits

# Model Building

## Init Params

In [64]:
learning_rate = 0.01
epochs = 50

In [65]:
model = SimpleRNN(vocab_size=len(vocabulary))

In [66]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## Training Loop/Pipeline

In [68]:
for epoch in range(epochs):

  losses = []
  model.train()

  for question, answer in dataloader:
     # Prediction
     y_pred = model(question)

     # Loss Calculate
     loss = criterion(y_pred, answer[0])
     losses.append(loss.item())

     # Backward
     optimizer.zero_grad()
     loss.backward()

     # Wright Update
     optimizer.step()

  avg_loss = np.mean(losses)
  print(f'Epochs: {epoch+1}, Loss: {avg_loss}')

Epochs: 1, Loss: 5.986849466959636
Epochs: 2, Loss: 3.745635085635715
Epochs: 3, Loss: 1.7543101776287788
Epochs: 4, Loss: 0.791920202691108
Epochs: 5, Loss: 0.5352753449283126
Epochs: 6, Loss: 0.37464208042042124
Epochs: 7, Loss: 0.30190169393188426
Epochs: 8, Loss: 0.24299266287125648
Epochs: 9, Loss: 0.14787068031728268
Epochs: 10, Loss: 0.1889157226619621
Epochs: 11, Loss: 0.10759004374138183
Epochs: 12, Loss: 0.08690552107420646
Epochs: 13, Loss: 0.1606750473069648
Epochs: 14, Loss: 0.06280738545918009
Epochs: 15, Loss: 0.10320431894441652
Epochs: 16, Loss: 0.10818975013939457
Epochs: 17, Loss: 0.01612955403721167
Epochs: 18, Loss: 0.007913364626518968
Epochs: 19, Loss: 0.005599096087583651
Epochs: 20, Loss: 0.0046918859071512185
Epochs: 21, Loss: 0.004054639097820553
Epochs: 22, Loss: 0.0036261877006230256
Epochs: 23, Loss: 0.003274744102317426
Epochs: 24, Loss: 0.002965482144564804
Epochs: 25, Loss: 0.0027196421378499103
Epochs: 26, Loss: 0.0025063539208430383
Epochs: 27, Loss: 

In [72]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indeces(question, vocabulary)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")
  else:
    print(list(vocabulary.keys())[index])

In [73]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [74]:
predict(model, "What is ML")

I don't know
